# Retail Sales & Customer Analytics Platform

# Phase 2: Data Integration

## Project Objective

The objective of this notebook is to combine all cleaned datasets into a single master analytical dataset.

The master dataset will be used for:

- PostgreSQL Database
- Advanced SQL Analysis
- Power BI Dashboard
- Business Reporting

-

## Workflow

1. Import Libraries
2. Load Cleaned Datasets
3. Validate Data
4. Merge Datasets
5. Validate Final Dataset
6. Export Master Dataset



In [28]:
# ==========================================================
# Step 1 : Import Required Libraries
# ==========================================================

# Data Manipulation
import pandas as pd
import numpy as np

# File Handling
from pathlib import Path

print("✅ Libraries Imported Successfully!")

✅ Libraries Imported Successfully!


In [29]:
# ==========================================================
# Step 2 : Set Project Directory
# ==========================================================

# Current notebook location
current_path = Path.cwd()

# Move to project root folder
project_path = current_path.parent

# Data folders
cleaned_data_path = project_path / "data" / "cleaned"

# Export folder
export_path = project_path / "exports"

print("Project Path:")
print(project_path)

print("\nCleaned Data Path:")
print(cleaned_data_path)

print("\nExport Path:")
print(export_path)

Project Path:
E:\Projects\Retail Sales & Customer Analytics Platform

Cleaned Data Path:
E:\Projects\Retail Sales & Customer Analytics Platform\data\cleaned

Export Path:
E:\Projects\Retail Sales & Customer Analytics Platform\exports


In [30]:
# ==========================================================
# Step 3 : Utility Function
# ==========================================================

def load_dataset(file_name):
    """
    Load a cleaned CSV dataset from the data/cleaned folder.

    Parameters
    ----------
    file_name : str
        Name of the CSV file.

    Returns
    -------
    pandas.DataFrame
        Loaded dataset.
    """

    file_path = cleaned_data_path / file_name

    dataframe = pd.read_csv(file_path)

    print(f"✅ Loaded: {file_name}")

    return dataframe

In [31]:
# ==========================================================
# Step 4 : Load All Cleaned Datasets
# ==========================================================

customers = load_dataset("customers_clean.csv")

orders = load_dataset("orders_clean.csv")

order_items = load_dataset("order_items_clean.csv")

products = load_dataset("products_clean.csv")

payments = load_dataset("payments_clean.csv")

reviews = load_dataset("reviews_clean.csv")

sellers = load_dataset("sellers_clean.csv")

geolocation = load_dataset("geolocation_clean.csv")

category_translation = load_dataset(
    "category_translation_clean.csv"
)

print("\n🎉 All datasets loaded successfully!")

✅ Loaded: customers_clean.csv
✅ Loaded: orders_clean.csv
✅ Loaded: order_items_clean.csv
✅ Loaded: products_clean.csv
✅ Loaded: payments_clean.csv
✅ Loaded: reviews_clean.csv
✅ Loaded: sellers_clean.csv
✅ Loaded: geolocation_clean.csv
✅ Loaded: category_translation_clean.csv

🎉 All datasets loaded successfully!


In [32]:
# ==========================================================
# Step 5 : Validate Loaded Datasets
# ==========================================================

datasets = {

    "Customers": customers,
    "Orders": orders,
    "Order Items": order_items,
    "Products": products,
    "Payments": payments,
    "Reviews": reviews,
    "Sellers": sellers,
    "Geolocation": geolocation,
    "Category Translation": category_translation

}

validation_summary = pd.DataFrame({

    "Dataset": datasets.keys(),

    "Rows": [df.shape[0] for df in datasets.values()],

    "Columns": [df.shape[1] for df in datasets.values()]

})

validation_summary

,Dataset,Rows,Columns
0,Customers,99441,5
1,Orders,99441,19
2,Order Items,112650,9
3,Products,32949,10
4,Payments,103886,5
5,Reviews,99224,8
6,Sellers,3095,4
7,Geolocation,1000163,5
8,Category Translation,71,2


## Merge Customers + Orders

In [34]:
# ==========================================================
# Step 6 : Merge Customers + Orders
# ==========================================================

orders_master = pd.merge(

    orders,

    customers,

    on="customer_id",

    how="left"

)

print("Merge Completed Successfully!")

print(f"\nRows    : {orders_master.shape[0]:,}")

print(f"Columns : {orders_master.shape[1]}")

Merge Completed Successfully!

Rows    : 99,441
Columns : 23


In [35]:
# ==========================================================
# Step 6 : Validation
# ==========================================================

missing_customers = (
    orders_master["customer_unique_id"]
    .isnull()
    .sum()
)

print(f"Missing Customer Records : {missing_customers}")

Missing Customer Records : 0


## Merge Order Items

In [36]:
# ==========================================================
# Step 7 : Merge Order Items
# ==========================================================

master_df = pd.merge(

    orders_master,

    order_items,

    on="order_id",

    how="left"

)

print("Merge Completed Successfully!")

print(f"\nRows    : {master_df.shape[0]:,}")

print(f"Columns : {master_df.shape[1]}")

Merge Completed Successfully!

Rows    : 113,425
Columns : 31


In [37]:
# ==========================================================
# Step 7 : Validation
# ==========================================================

missing_products = (
    master_df["product_id"]
    .isnull()
    .sum()
)

print(f"Orders Without Products : {missing_products}")

Orders Without Products : 775


## Merge Products

In [40]:
# ==========================================================
# Step 8 : Merge Products
# ==========================================================

master_df = pd.merge(

    master_df,

    products,

    on="product_id",

    how="left"

)

print("Merge Completed Successfully!")

print(f"\nRows    : {master_df.shape[0]:,}")

print(f"Columns : {master_df.shape[1]}")


# ==========================================================
# Step 8 : Validation
# ==========================================================

missing_product_info = (
    master_df["product_category_name"]
    .isnull()
    .sum()
)


Merge Completed Successfully!

Rows    : 113,425
Columns : 40


## Merge Category Translation

In [41]:
# ==========================================================
# Step 9 : Merge Category Translation
# ==========================================================

master_df = pd.merge(

    master_df,

    category_translation,

    on="product_category_name",

    how="left"

)

print("Merge Completed Successfully!")

print(f"\nRows    : {master_df.shape[0]:,}")

print(f"Columns : {master_df.shape[1]}")

print(
    "Missing English Categories :",
    master_df["product_category_name_english"]
    .isnull()
    .sum()
)

Merge Completed Successfully!

Rows    : 113,425
Columns : 41
Missing English Categories : 2403


In [42]:
# ==========================================================
# Step 10.1 : Categories Without Translation
# ==========================================================

missing_categories = (
    master_df[
        master_df["product_category_name_english"].isna()
    ][["product_category_name"]]
    .drop_duplicates()
)

missing_categories

,product_category_name
6,Unknown
306,NaN
1045,portateis_cozinha_e_preparadores_de_alimentos
2515,pc_gamer


In [43]:
# ==========================================================
# Count Missing Categories
# ==========================================================

master_df.loc[
    master_df["product_category_name_english"].isna(),
    "product_category_name"
].value_counts()

product_category_name
Unknown                                          1586
portateis_cozinha_e_preparadores_de_alimentos      15
pc_gamer                                            9
Name: count, dtype: int64

In [44]:
# ==========================================================
# Step 10.3 : Handle Missing English Categories
# ==========================================================

master_df["product_category_name_english"] = (
    master_df["product_category_name_english"]
    .fillna("Unknown")
)

print("Missing English Categories :")

print(
    master_df["product_category_name_english"]
    .isnull()
    .sum()
)

Missing English Categories :
0


In [45]:
# ==========================================================
# Step 11 : Merge Sellers
# ==========================================================

master_df = merge_data(
    master_df,
    sellers,
    "seller_id"
)

# ==========================================================
# Step 11 : Validate Seller Merge
# ==========================================================

missing_sellers = (
    master_df["seller_city"]
    .isnull()
    .sum()
)

missing_percentage = (
    missing_sellers /
    len(master_df)
) * 100

print(f"Missing Seller Information : {missing_sellers:,}")
print(f"Missing Percentage         : {missing_percentage:.2f}%")

✅ Merge Completed Successfully!
Rows    : 113,425
Columns : 44
Missing Seller Information : 775
Missing Percentage         : 0.68%


In [46]:
# ==========================================================
# Utility Function : Merge Two DataFrames
# ==========================================================

def merge_data(left_df, right_df, key, how="left"):
    """
    Merge two DataFrames and display merge summary.

    Parameters
    ----------
    left_df : pandas.DataFrame
        Left DataFrame.

    right_df : pandas.DataFrame
        Right DataFrame.

    key : str
        Column name used for joining.

    how : str
        Type of join (default = left).

    Returns
    -------
    pandas.DataFrame
        Merged DataFrame.
    """

    merged_df = pd.merge(
        left_df,
        right_df,
        on=key,
        how=how
    )

    print("✅ Merge Completed Successfully!")
    print(f"Rows    : {merged_df.shape[0]:,}")
    print(f"Columns : {merged_df.shape[1]}")

    return merged_df

In [47]:
# ==========================================================
# Utility Function : Validate Merge
# ==========================================================

def validate_merge(df, column_name):
    """
    Check missing values after a merge.

    Parameters
    ----------
    df : pandas.DataFrame
        Merged DataFrame.

    column_name : str
        Column used for validation.
    """

    missing = df[column_name].isna().sum()

    percentage = (
        missing / len(df)
    ) * 100

    print(f"Missing {column_name}: {missing:,}")

    print(f"Missing Percentage : {percentage:.2f}%")

In [48]:
# ==========================================================
# Step 11 : Merge Sellers
# ==========================================================

master_df = merge_data(
    master_df,
    sellers,
    "seller_id"
)

✅ Merge Completed Successfully!
Rows    : 113,425
Columns : 47


In [56]:
# ==========================================================
# Step 11 : Validate Seller Merge
# ==========================================================

validate_merge(
    master_df,
    "seller_city"
)

Missing seller_city: 775
Missing Percentage : 0.68%


In [50]:
master_df.columns.tolist()

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'order_year_month',
 'order_day',
 'order_month',
 'delivery_time_days',
 'approval_time_hours',
 'delivery_delay_days',
 'estimated_vs_actual_delivery',
 'order_year',
 'order_quarter',
 'is_delayed',
 'delivery_status',
 'customer_unique_id',
 'customer_zip_code_prefix',
 'customer_city',
 'customer_state',
 'order_item_id',
 'product_id',
 'seller_id',
 'shipping_limit_date',
 'price',
 'freight_value',
 'total_item_cost',
 'freight_percentage',
 'product_category_name',
 'product_name_lenght',
 'product_description_lenght',
 'product_photos_qty',
 'product_weight_g',
 'product_length_cm',
 'product_height_cm',
 'product_width_cm',
 'product_volume_cm3',
 'product_category_name_english',
 'seller_zip_code_prefix_x',
 'seller_city_x',
 'seller_state_x',
 'seller_zip_code_prefix_y',
 'sell

In [51]:
pd.Series(master_df.columns)

0                          order_id
1                       customer_id
2                      order_status
3          order_purchase_timestamp
4                 order_approved_at
5      order_delivered_carrier_date
6     order_delivered_customer_date
7     order_estimated_delivery_date
8                  order_year_month
9                         order_day
10                      order_month
11               delivery_time_days
12              approval_time_hours
13              delivery_delay_days
14     estimated_vs_actual_delivery
15                       order_year
16                    order_quarter
17                       is_delayed
18                  delivery_status
19               customer_unique_id
20         customer_zip_code_prefix
21                    customer_city
22                   customer_state
23                    order_item_id
24                       product_id
25                        seller_id
26              shipping_limit_date
27                          

In [52]:
print(master_df.shape)
print(sellers.columns)

(113425, 47)
Index(['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state'], dtype='object')


In [53]:
master_df[[
    "seller_city_x",
    "seller_city_y"
]].head()

,seller_city_x,seller_city_y
0,maua,maua
1,belo horizonte,belo horizonte
2,guariba,guariba
3,belo horizonte,belo horizonte
4,mogi das cruzes,mogi das cruzes


In [54]:
# ==========================================================
# Remove Duplicate Seller Columns
# ==========================================================

master_df = master_df.drop(
    columns=[
        "seller_zip_code_prefix_y",
        "seller_city_y",
        "seller_state_y"
    ]
)

master_df = master_df.rename(columns={
    "seller_zip_code_prefix_x": "seller_zip_code_prefix",
    "seller_city_x": "seller_city",
    "seller_state_x": "seller_state"
})

print("✅ Duplicate seller columns removed.")

✅ Duplicate seller columns removed.


In [55]:
validate_merge(
    master_df,
    "seller_city"
)

Missing seller_city: 775
Missing Percentage : 0.68%


In [57]:
# ==========================================================
# Step 12 : Merge Payments
# ==========================================================

master_df = merge_data(
    master_df,
    payments,
    "order_id"
)

✅ Merge Completed Successfully!
Rows    : 118,434
Columns : 48


In [59]:
# ==========================================================
# Step 12 : Validate Payment Merge
# ==========================================================

validate_merge(
    master_df,
    "payment_type"
)

print("\nPayment Records")

print(master_df["payment_type"].value_counts(dropna=False))

Missing payment_type: 3
Missing Percentage : 0.00%

Payment Records
payment_type
credit_card    87286
boleto         23037
voucher         6407
debit_card      1698
not_defined        3
NaN                3
Name: count, dtype: int64


In [61]:
# ==========================================================
# Step 13 : Merge Reviews
# ==========================================================

master_df = merge_data(
    master_df,
    reviews,
    "order_id"
)

# ==========================================================
# Step 13 : Validate Review Merge
# ==========================================================

validate_merge(
    master_df,
    "review_score"
)

✅ Merge Completed Successfully!
Rows    : 120,573
Columns : 62


KeyError: 'review_score'

In [62]:
# ==========================================================
# Step 14 : Master Dataset Overview
# ==========================================================

print("=" * 50)
print("MASTER DATASET SUMMARY")
print("=" * 50)

print(f"Rows    : {master_df.shape[0]:,}")
print(f"Columns : {master_df.shape[1]}")

print("\nData Types")

print(master_df.dtypes.value_counts())

MASTER DATASET SUMMARY
Rows    : 120,573
Columns : 62

Data Types
object     36
float64    23
int64       3
Name: count, dtype: int64


In [63]:
# ==========================================================
# Step 15 : Missing Value Summary
# ==========================================================

missing_summary = (
    master_df.isnull()
    .sum()
    .sort_values(ascending=False)
)

missing_summary = missing_summary[
    missing_summary > 0
]

missing_summary = missing_summary.to_frame(
    name="Missing Values"
)

missing_summary["Missing Percentage"] = (
    missing_summary["Missing Values"] /
    len(master_df)
) * 100

missing_summary

,Missing Values,Missing Percentage
review_comment_title_x,106544,88.364725
review_comment_title_y,106544,88.364725
review_comment_message_y,69808,57.896876
review_comment_message_x,69808,57.896876
delivery_time_days,3469,2.877095
order_delivered_customer_date,3469,2.877095
delivery_delay_days,3469,2.877095
estimated_vs_actual_delivery,3469,2.877095
product_name_lenght,2571,2.132318
product_description_lenght,2571,2.132318


In [64]:
# ==========================================================
# Step 16 : Duplicate Records
# ==========================================================

duplicates = master_df.duplicated().sum()

print(f"Duplicate Rows : {duplicates:,}")

Duplicate Rows : 0


In [65]:
print(reviews.columns.tolist())

['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp', 'review_category']


In [66]:
[col for col in master_df.columns if "review" in col.lower()]

['review_id_x',
 'review_score_x',
 'review_comment_title_x',
 'review_comment_message_x',
 'review_creation_date_x',
 'review_answer_timestamp_x',
 'review_category_x',
 'review_id_y',
 'review_score_y',
 'review_comment_title_y',
 'review_comment_message_y',
 'review_creation_date_y',
 'review_answer_timestamp_y',
 'review_category_y']

In [67]:
print(reviews.shape)

(99224, 8)


In [68]:
master_df[
    ["review_score_x", "review_score_y"]
].head(10)

,review_score_x,review_score_y
0,4.0,4.0
1,4.0,4.0
2,4.0,4.0
3,4.0,4.0
4,5.0,5.0
5,5.0,5.0
6,5.0,5.0
7,4.0,4.0
8,2.0,2.0
9,5.0,5.0


In [69]:
# ==========================================================
# Remove Duplicate Review Columns
# ==========================================================

# Drop the duplicate (_y) columns
master_df = master_df.drop(columns=[
    "review_id_y",
    "review_score_y",
    "review_comment_title_y",
    "review_comment_message_y",
    "review_creation_date_y",
    "review_answer_timestamp_y",
    "review_category_y"
])

# Rename the original (_x) columns
master_df = master_df.rename(columns={
    "review_id_x": "review_id",
    "review_score_x": "review_score",
    "review_comment_title_x": "review_comment_title",
    "review_comment_message_x": "review_comment_message",
    "review_creation_date_x": "review_creation_date",
    "review_answer_timestamp_x": "review_answer_timestamp",
    "review_category_x": "review_category"
})

print("✅ Duplicate review columns removed successfully!")

✅ Duplicate review columns removed successfully!


In [70]:
validate_merge(master_df, "review_score")

Missing review_score: 997
Missing Percentage : 0.83%


In [71]:
duplicates = [
    col for col in master_df.columns
    if col.endswith("_x") or col.endswith("_y")
]

print(duplicates)

[]


In [72]:
# ==========================================================
# Final Validation 1 : Master Dataset Overview
# ==========================================================

print("=" * 60)
print("MASTER DATASET SUMMARY")
print("=" * 60)

print(f"Rows    : {master_df.shape[0]:,}")
print(f"Columns : {master_df.shape[1]}")

print("\nData Types")

print(master_df.dtypes.value_counts())

MASTER DATASET SUMMARY
Rows    : 120,573
Columns : 55

Data Types
object     30
float64    22
int64       3
Name: count, dtype: int64


In [73]:
# ==========================================================
# Final Validation 2 : Missing Value Summary
# ==========================================================

missing_summary = (
    master_df.isnull()
    .sum()
    .sort_values(ascending=False)
)

missing_summary = missing_summary[
    missing_summary > 0
]

missing_summary = missing_summary.to_frame(
    name="Missing Values"
)

missing_summary["Missing Percentage"] = (
    missing_summary["Missing Values"]
    / len(master_df)
) * 100

missing_summary

,Missing Values,Missing Percentage
review_comment_title,106544,88.364725
review_comment_message,69808,57.896876
order_delivered_customer_date,3469,2.877095
delivery_time_days,3469,2.877095
delivery_delay_days,3469,2.877095
estimated_vs_actual_delivery,3469,2.877095
product_description_lenght,2571,2.132318
product_photos_qty,2571,2.132318
product_name_lenght,2571,2.132318
order_delivered_carrier_date,2110,1.749977


In [74]:
# ==========================================================
# Final Validation 3 : Duplicate Rows
# ==========================================================

duplicate_rows = master_df.duplicated().sum()

print(f"Duplicate Rows : {duplicate_rows:,}")

Duplicate Rows : 1,430


In [75]:
# ==========================================================
# Final Validation 4 : Duplicate Column Names
# ==========================================================

duplicate_columns = master_df.columns[
    master_df.columns.duplicated()
]

if len(duplicate_columns) == 0:
    print("✅ No duplicate column names found.")
else:
    print("Duplicate Columns:")
    print(duplicate_columns.tolist())

✅ No duplicate column names found.


In [76]:
# ==========================================================
# Final Validation 5 : Merge Suffix Check
# ==========================================================

merge_suffix_columns = [
    column
    for column in master_df.columns
    if column.endswith("_x") or column.endswith("_y")
]

if len(merge_suffix_columns) == 0:
    print("✅ No merge suffix columns found.")
else:
    print("Merge Suffix Columns:")
    print(merge_suffix_columns)

✅ No merge suffix columns found.


In [77]:
# ==========================================================
# Final Validation 6 : Data Quality Report
# ==========================================================

data_quality_report = pd.DataFrame({
    "Metric": [
        "Total Rows",
        "Total Columns",
        "Duplicate Rows",
        "Columns with Missing Values"
    ],
    "Value": [
        len(master_df),
        master_df.shape[1],
        master_df.duplicated().sum(),
        (master_df.isnull().sum() > 0).sum()
    ]
})

data_quality_report

,Metric,Value
0,Total Rows,120573
1,Total Columns,55
2,Duplicate Rows,1430
3,Columns with Missing Values,38


In [78]:
# ==========================================================
# Final Validation 7 : Sample Records
# ==========================================================

master_df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_year_month,order_day,...,payment_type,payment_installments,payment_value,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,review_category
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,2017-10,Monday,...,credit_card,1.0,18.12,a54f0611adc9ed256b57ede6b6eb5114,4.0,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11 00:00:00,2017-10-12 03:43:48,Positive
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,2017-10,Monday,...,voucher,1.0,2.00,a54f0611adc9ed256b57ede6b6eb5114,4.0,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11 00:00:00,2017-10-12 03:43:48,Positive
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,2017-10,Monday,...,voucher,1.0,18.59,a54f0611adc9ed256b57ede6b6eb5114,4.0,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11 00:00:00,2017-10-12 03:43:48,Positive
3,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,2018-07,Tuesday,...,boleto,1.0,141.46,8d5266042046a06655c8db133d120ba5,4.0,Muito boa a loja,Muito bom o produto.,2018-08-08 00:00:00,2018-08-08 18:37:50,Positive
4,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,2018-08,Wednesday,...,credit_card,3.0,179.12,e73b67b67587f7644d5bd1a52deb1b01,5.0,NaN,NaN,2018-08-18 00:00:00,2018-08-22 19:07:58,Positive


In [79]:
# ==========================================================
# Final Step : Export Master Dataset
# ==========================================================

import os

output_folder = "../data/processed"

os.makedirs(output_folder, exist_ok=True)

# Export CSV
master_df.to_csv(
    os.path.join(output_folder, "master_dataset.csv"),
    index=False
)

# Export Excel
master_df.to_excel(
    os.path.join(output_folder, "master_dataset.xlsx"),
    index=False
)

print("✅ Master Dataset exported successfully!")

✅ Master Dataset exported successfully!
